# TradingAgents #1275: FRED serves today's vintage to a historical run

https://github.com/TauricResearch/TradingAgents/issues/1275

`get_macro_data` bounded FRED requests by observation date only. The vintage window
(`realtime_start`/`realtime_end`) defaults to today, so a run dated 2023-03-31 received March CPI,
released 2023-04-12, at its revised value.

The frame below has the shape of `fredapi.Fred.get_series_all_releases("CPIAUCSL")`. Release
dates for the February and March 2023 prints are the real ones; the values are synthetic.

In [1]:
import pandas as pd
import fin_pit as pit
from fin_pit.adapters import fred

raw = pd.DataFrame({
    "date": pd.to_datetime(["2023-01-01", "2023-02-01", "2023-03-01", "2023-01-01", "2023-02-01", "2023-03-01"]),
    "realtime_start": pd.to_datetime(["2023-02-14", "2023-03-14", "2023-04-12", "2024-02-13", "2024-02-13", "2024-02-13"]),
    "value": [299.2, 300.8, 301.8, 299.0, 300.5, 301.6],   # synthetic
})

## The leak: observation-date bound, today's vintage

In [2]:
latest = raw.sort_values("realtime_start").groupby("date").tail(1)
latest[latest["date"] <= "2023-03-31"].sort_values("date")

,date,realtime_start,value
3,2023-01-01,2024-02-13,299.0
4,2023-02-01,2024-02-13,300.5
5,2023-03-01,2024-02-13,301.6


## With fin-pit: vintages stamped with their release, read as of the decision

In [3]:
stamped = fred.from_fredapi(raw, "CPIAUCSL")
with pit.asof("2023-03-31 16:00", tz="America/New_York"):
    cpi = fred.asof_series(stamped)
cpi

event_time
2023-01-01    299.2
2023-02-01    300.8
Name: CPIAUCSL, dtype: float64

In [4]:
assert list(cpi.index.strftime("%Y-%m-%d")) == ["2023-01-01", "2023-02-01"]
assert cpi.tolist() == [299.2, 300.8]
fred.observation_params(pit.to_utc("2023-03-31 16:00", "America/New_York"))   # for raw API calls

{'realtime_start': '2023-03-30', 'realtime_end': '2023-03-30'}